# Personal Expense Tracker - Exploratory Analysis
This notebook walks through the **data-analyst view** of the project: load → inspect → clean → analyse → visualise.

> Run `python main.py run-all` once from the project root first (it builds `db/expenses.db`), then run this notebook top-to-bottom.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))     # project root (notebook lives in notebooks/)

import pandas as pd
from src import analyze, budget, plots, ingest, config
from src.config import inr
pd.options.display.float_format = '{:,.2f}'.format

## 1. Load the data from SQLite

In [ ]:
df = analyze.fetch_frame()
print(f"{len(df):,} transactions | {df.tx_date.min():%d %b %Y} -> {df.tx_date.max():%d %b %Y} | accounts: {df.account.unique().tolist()}")
df.head(10)

## 2. Data quality checks

In [ ]:
print(df.isna().sum()[lambda s: s > 0])           # missing values per column
print('Duplicates :', df.duplicated(['tx_date','description','amount','account']).sum())
print('Uncategorized rows:', (df.category == 'Uncategorized').sum())
df.groupby('category_source').size()

### Cleaning demo - the messy bank statement
The raw HDFC-style export has different headers, `dd/mm/yy` dates, split debit/credit columns, a duplicated row, a blank-amount row and a footer.

In [ ]:
raw = pd.read_csv(config.BANK_CSV, dtype=str)
raw.tail(6)

In [ ]:
clean, report = ingest.standardize(raw, account='HDFC Savings')
print(report)
clean.head()

## 3. Key numbers

In [ ]:
month = analyze.default_month(df)
k = analyze.kpis(analyze.filter_month(df, month), analyze.period_days(df, month))
print(f"Month {month}: spend {inr(k['total_spend'])}, income {inr(k['income'])}, savings {inr(k['savings'])} ({k['savings_rate']:.0f}%)")
print('Highest category:', k['top_category'], inr(k['top_category_amount']))
analyze.category_summary(df, month)

## 4. Category-wise spending

In [ ]:
plots.plot_category_bar(df, month)

## 5. Monthly trend (income vs spending)

In [ ]:
display(analyze.monthly_summary(df).round(1))
plots.plot_monthly_trend(df, highlight=month)

## 6. Payment methods

In [ ]:
display(analyze.payment_summary(df, month))
plots.plot_payment_donut(df, month)

## 7. Daily spending and weekday pattern

In [ ]:
plots.plot_daily_trend(df, month)

In [ ]:
plots.plot_weekday_heatmap(df, month)

## 8. Budgets

In [ ]:
status = budget.check_budget(month)
display(status.round(1))
plots.plot_budget_vs_actual(status, month)

## 9. Recurring payments, anomalies and forecast

In [ ]:
analyze.detect_recurring(df)

In [ ]:
analyze.detect_anomalies(df)

In [ ]:
fc = analyze.forecast_next_month(df)
print('Next month:', fc['next_month'], '| expected spend', inr(fc['total']), '| trend check', inr(fc['trend_total']))
fc['by_category']

## 10. Plain-English insights

In [ ]:
for line in analyze.generate_insights(df, month, status):
    print('•', line)

## Exercises
1. Change `month` to another month and re-run.
2. Add a new rule to `src/categorize.py` and re-run `python main.py categorize --all`.
3. Lower the anomaly threshold (`min_multiple=3`) and see which extra transactions appear.